# ResNet18 Pretrained Transfer Learning

Purpose: collect the ResNet18 pretrained runs in one notebook. This includes the frozen feature extractor baseline, the augmented frozen run, and the last-block fine-tuning runs on the stricter split seeds.

This notebook reads from `reports/tables/model_results_master.csv`; it does not duplicate the training code.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

master_path = ROOT / 'reports' / 'tables' / 'model_results_master.csv'
summary_path = ROOT / 'reports' / 'tables' / 'model_results_master.json'
master = pd.read_csv(master_path)
summary = json.loads(summary_path.read_text(encoding='utf-8'))

metric_cols = [
    'model_name', 'run_group', 'pretrained', 'seed', 'split',
    'accuracy', 'precision_macro', 'recall_macro', 'f1_macro',
    'best_epoch', 'epochs_trained', 'recommendation', 'notes'
]

In [ ]:
resnet_pretrained = master[(master['model_family'] == 'resnet18') & (master['pretrained'] == True)].copy()
resnet_pretrained[metric_cols].sort_values(['run_group', 'seed'])

In [ ]:
resnet_pretrained.groupby('run_group').agg(
    runs=('accuracy', 'count'),
    mean_accuracy=('accuracy', 'mean'),
    min_accuracy=('accuracy', 'min'),
    max_accuracy=('accuracy', 'max'),
    mean_f1=('f1_macro', 'mean'),
).reset_index()

## Reproduction Commands

The main strict-split transfer-learning runs are produced with:

```powershell
python -m src.data.create_strict_split_manifests --seeds 42 123 999
python -m src.training.train_resnet18_finetune_last_block --manifest reports/tables/strict_split_manifest_seed42.csv --seed 42 --artifact-prefix resnet18_finetune_last_block_strict_seed42
python -m src.training.train_resnet18_finetune_last_block --manifest reports/tables/strict_split_manifest_seed123.csv --seed 123 --artifact-prefix resnet18_finetune_last_block_strict_seed123
python -m src.training.train_resnet18_finetune_last_block --manifest reports/tables/strict_split_manifest_seed999.csv --seed 999 --artifact-prefix resnet18_finetune_last_block_strict_seed999
```